Task 2
Data Engineering

Plan:
(A) validate every booking against a Pydantic contract and quarantine the bad ones, including resent duplicates
(B) flatten the surviving nested documents into two flat tables
(C) model them as a star schema in DuckDB and save it as Parquet.

Every number below is computed from DATA. Nothing is typed in by hand.

In [1]:
DATA = "bookings.jsonl"

In [2]:
import datetime as dt
import json
import math
import shutil
from collections import Counter
from pathlib import Path
from typing import Literal

import duckdb
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

Part A: contract

we have four models: Ticket, Student, Event, and Booking which uses the other three.
Field level rules (formats, allowed values, qty >= 1, price >= 0, at least one ticket, event present) are declared on the fields.
The two rules that involve more than one field (total = sum of qty × price and booked_at before the event date) are in a model_validator.

extra="allow" on Booking, Student and Event  means a field that is not in the contract does NOT make a booking invalid, it is kept (that is how schema drift is handled in Part B).

In [ ]:
FACULTIES = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
CLUBS = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TICKET_TYPES = Literal["standard", "vip", "student"]
PAYMENTS = Literal["card", "cash", "wallet"]
STATUSES = Literal["confirmed", "cancelled"]


class Ticket(BaseModel):
    type: TICKET_TYPES
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str = Field(pattern=r"^[0-9]{2}-[0-9]{4}$")   # e.g. 58-0275
    name: str
    faculty: FACULTIES


class Event(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str
    title: str
    club: CLUBS
    date: dt.date
    venue: str


class Booking(BaseModel):
    model_config = ConfigDict(extra="allow")
    booking_id: str
    booked_at: dt.datetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: PAYMENTS
    status: STATUSES

    @model_validator(mode="after")
    def check_cross_field_rules(self):
        # Rule: total equals the sum of qty x price (tolerance of half a piastre for float noise)
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=0.005):
            raise ValueError("total does not equal sum of qty x price")
        # Rule: booked_at (UTC) must be before the event's date
        booked = self.booked_at
        if booked.tzinfo is not None:
            booked = booked.astimezone(dt.timezone.utc)
        if booked.date() >= self.event.date:
            raise ValueError("booked_at is not before the event date")
        return self

Validate every line
validate_doc returns the model or a list of human readable reasons. A booking can break several rules, so we keep all reasons, and use the first one as its main reason for the counts.
Lines that are not JSON at all, or not a JSON object, are quarantined too (they have no booking_id).

In [4]:
def error_label(err):
    """Turn one Pydantic error into a short, stable reason (no row-specific numbers, so reasons can be counted)."""
    path = ".".join(str(p) for p in err["loc"] if not isinstance(p, int))   # drop list indexes
    kind = err["type"]
    if kind == "value_error":                                               # our model_validator messages
        return err["msg"].removeprefix("Value error, ")
    if kind == "missing" or err["input"] is None:
        return f"{path}: missing or null"
    if kind == "literal_error":
        return f"{path}: not an allowed value"
    if kind == "string_pattern_mismatch":
        return f"{path}: wrong format"
    if kind == "greater_than_equal":
        return f"{path}: below the minimum ({err['ctx']['ge']})"
    if kind == "too_short":
        return f"{path}: needs at least {err['ctx']['min_length']}"
    return f"{path}: {err['msg']}"


def validate_doc(doc):
    """Return (Booking or None, [reasons])."""
    if not isinstance(doc, dict):
        return None, ["line is not a JSON object"]
    try:
        return Booking.model_validate(doc), []
    except ValidationError as e:
        reasons = list(dict.fromkeys(error_label(x) for x in e.errors()))
        return None, reasons


def booking_id_of(doc):
    bid = doc.get("booking_id") if isinstance(doc, dict) else None
    return bid if isinstance(bid, (str, int)) else None


# Read DATA: one record per non-blank line
records = []
with open(DATA, encoding="utf-8") as f:
    for line_no, text in enumerate(f, start=1):
        if not text.strip():
            continue
        try:
            doc = json.loads(text)
        except json.JSONDecodeError:
            doc = None
            records.append({"line": line_no, "doc": text.rstrip("\n"), "json_ok": False})
            continue
        records.append({"line": line_no, "doc": doc, "json_ok": True})

for r in records:
    if r["json_ok"]:
        r["model"], r["reasons"] = validate_doc(r["doc"])
    else:
        r["model"], r["reasons"] = None, ["line is not valid JSON"]
    r["booking_id"] = booking_id_of(r["doc"]) if r["json_ok"] else None

print(f"{len(records)} lines read from {DATA}")

283 lines read from bookings.jsonl


Resent bookings (duplicates)
The duplicate check runs after validation, on the bookings that passed. The first copy of each booking_id (in file order) is kept & later copies go to the quarantine as duplicate booking_id.

In [5]:
quarantine, valid, seen = [], [], set()

for r in records:                                   # file order = "first copy wins"
    if r["model"] is None:
        quarantine.append({"booking_id": r["booking_id"], "reason": r["reasons"][0],
                           "all_reasons": r["reasons"], "line": r["line"], "raw": r["doc"]})
    elif r["booking_id"] in seen:
        quarantine.append({"booking_id": r["booking_id"], "reason": "duplicate booking_id",
                           "all_reasons": ["duplicate booking_id"], "line": r["line"], "raw": r["doc"]})
    else:
        seen.add(r["booking_id"])
        valid.append(r["model"])
        r["kept"] = True

with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print(f"quarantine.jsonl written: {len(quarantine)} rows")

quarantine.jsonl written: 24 rows


Report

In [6]:
n_read = len(records)
n_invalid = sum(1 for r in records if r["model"] is None)
n_dupes = sum(1 for q in quarantine if q["reason"] == "duplicate booking_id")

print(f"Lines read:                       {n_read}")
print(f"Passed validation (before dedupe): {n_read - n_invalid}")
print(f"Quarantined:                      {len(quarantine)}  ({n_invalid} failed validation + {n_dupes} duplicates)")
print(f"Passed (valid and unique):        {len(valid)}")
assert n_read == len(valid) + len(quarantine)

reason_counts = pd.Series(Counter(q["reason"] for q in quarantine), name="bookings", dtype="int64")
print("\nQuarantined, by reason:")
display(reason_counts.sort_values(ascending=False).to_frame())

multi = sum(1 for q in quarantine if len(q["all_reasons"]) > 1)
print(f"{multi} quarantined bookings broke more than one rule (each is counted once, under its first reason; all reasons are in quarantine.jsonl)")

Lines read:                       283
Passed validation (before dedupe): 262
Quarantined:                      24  (21 failed validation + 3 duplicates)
Passed (valid and unique):        259

Quarantined, by reason:


,bookings
student.id: wrong format,4
total does not equal sum of qty x price,4
tickets.qty: below the minimum (1),3
tickets.type: not an allowed value,3
duplicate booking_id,3
booked_at is not before the event date,3
event: missing or null,2
tickets: needs at least 1,2


0 quarantined bookings broke more than one rule (each is counted once, under its first reason; all reasons are in quarantine.jsonl)


Why duplicates come after validation, what would change if we swapped?
We replay the pipeline in the opposite order (deduplicate the raw lines first, then validate) and compare with the real result.

In [7]:
def reason_counts_for(order):
    out, seen_ids, kept = Counter(), set(), 0
    if order == "validate_then_dedupe":
        for r in records:
            if r["model"] is None:
                out[r["reasons"][0]] += 1
            elif r["booking_id"] in seen_ids:
                out["duplicate booking_id"] += 1
            else:
                seen_ids.add(r["booking_id"]); kept += 1
    else:   # dedupe_then_validate: first line with an id wins, whether or not it is valid
        for r in records:
            if r["booking_id"] is not None and r["booking_id"] in seen_ids:
                out["duplicate booking_id"] += 1
            else:
                if r["booking_id"] is not None:
                    seen_ids.add(r["booking_id"])
                if r["model"] is None:
                    out[r["reasons"][0]] += 1
                else:
                    kept += 1
    return out, kept

a, kept_a = reason_counts_for("validate_then_dedupe")
b, kept_b = reason_counts_for("dedupe_then_validate")
cmp_ = pd.DataFrame({"validate -> dedupe": pd.Series(a), "dedupe -> validate": pd.Series(b)}).fillna(0).astype(int)
cmp_.loc["bookings kept"] = [kept_a, kept_b]
display(cmp_)
print("Counts identical?", bool((cmp_.iloc[:, 0] == cmp_.iloc[:, 1]).all()))

# Duplicate ids where the first copy is INVALID and a later copy is valid: swapping would throw the good copy away
by_id = {}
for r in records:
    if r["booking_id"] is not None:
        by_id.setdefault(r["booking_id"], []).append(r)
lost = [i for i, rs in by_id.items() if len(rs) > 1 and rs[0]["model"] is None and any(x["model"] for x in rs[1:])]
print("booking_ids whose first copy is invalid but a later copy is valid:", lost)

,validate -> dedupe,dedupe -> validate
tickets.type: not an allowed value,3,3
tickets.qty: below the minimum (1),3,3
student.id: wrong format,4,4
duplicate booking_id,3,3
event: missing or null,2,2
total does not equal sum of qty x price,4,4
booked_at is not before the event date,3,3
tickets: needs at least 1,2,2
bookings kept,259,259


Counts identical? True
booking_ids whose first copy is invalid but a later copy is valid: []


Part B: Flatten

From the valid bookings:
    -bookings: one row per booking, student and event flattened into student_* / event_* columns, no tickets column.
    -booking_tickets: one row per ticket entry, with amount = qty × price.

In [8]:
CORE = ["booking_id", "booked_at",
        "student_id", "student_name", "student_faculty",
        "event_id", "event_title", "event_club", "event_date", "event_venue",
        "total", "payment", "status"]

flat = pd.json_normalize([b.model_dump() for b in valid], sep="_").drop(columns="tickets")
extra_cols = [c for c in flat.columns if c not in CORE]          # drifted fields (kept, NULL where absent)
bookings = flat[CORE + extra_cols].copy()
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True)
bookings["event_date"] = pd.to_datetime(bookings["event_date"])

booking_tickets = pd.DataFrame(
    [{"booking_id": b.booking_id, "type": t.type, "qty": t.qty, "price": t.price, "amount": t.qty * t.price}
     for b in valid for t in b.tickets]
)

print("bookings:", bookings.shape, "| booking_tickets:", booking_tickets.shape)
display(bookings.head(3))
display(booking_tickets.head(3))

bookings: (259, 14) | booking_tickets: (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00+00:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,NaN
1,B0002,2026-09-27 12:29:00+00:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,NaN
2,B0003,2026-10-13 20:19:00+00:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,NaN


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0


Checks

In [9]:
# 1. booking_id is unique in bookings
assert bookings["booking_id"].is_unique, "duplicate booking_id in bookings"

# 2. every ticket belongs to a booking, and the amounts of a booking's tickets add up to its total
assert booking_tickets["booking_id"].isin(bookings["booking_id"]).all()
amount_per_booking = booking_tickets.groupby("booking_id")["amount"].sum().reindex(bookings["booking_id"])
assert np.isclose(amount_per_booking.to_numpy(), bookings["total"].to_numpy(), atol=0.005, rtol=0).all(), \
    "sum of ticket amounts differs from total for some booking"
print("OK: booking_id unique; ticket amounts add up to total for all", len(bookings), "bookings")

OK: booking_id unique; ticket amounts add up to total for all 259 bookings


Schema drift
We compare the keys that actually occur in the valid documents with the keys the contract knows (taken from the models themselves) and report every extra key, how many documents have it and the earliest booked_at at which it shows up.
The file is not in time order, so "from when" is the minimum booked_at not the first line.

In [10]:
def key_paths(doc):
    paths = set(doc) - {"student", "event", "tickets"}
    paths |= {f"student.{k}" for k in doc.get("student", {})}
    paths |= {f"event.{k}" for k in doc.get("event", {})}
    for t in doc.get("tickets", []):
        paths |= {f"tickets[].{k}" for k in t}
    return paths

known = (set(Booking.model_fields) - {"student", "event", "tickets"}) \
    | {f"student.{k}" for k in Student.model_fields} \
    | {f"event.{k}" for k in Event.model_fields} \
    | {f"tickets[].{k}" for k in Ticket.model_fields}

kept_lines = {r["booking_id"]: r for r in records if r.get("kept")}
rows = []
for bid, r in kept_lines.items():
    for p in key_paths(r["doc"]) - known:
        rows.append({"field": p, "booking_id": bid, "booked_at": r["model"].booked_at})
drift = pd.DataFrame(rows, columns=["field", "booking_id", "booked_at"])

if drift.empty:
    print("No schema drift: every valid document has exactly the contract's fields.")
else:
    summary = []
    for field, g in drift.groupby("field"):
        first = g["booked_at"].min()
        after = sum(1 for r in kept_lines.values()
                    if r["model"].booked_at >= first and field not in key_paths(r["doc"]))
        summary.append({"field": field, "docs_with_field": len(g), "docs_total": len(kept_lines),
                        "first_booked_at": first, "first_booking_id": g.loc[g["booked_at"].idxmin(), "booking_id"],
                        "docs_without_it_since_then": after})
    drift_summary = pd.DataFrame(summary)
    display(drift_summary)
    for _, d in drift_summary.iterrows():
        print(f"'{d.field}' is in {d.docs_with_field} of {d.docs_total} valid bookings, first appearing on {d.first_booked_at:%Y-%m-%d} ({d.first_booked_at:%H:%M} UTC, {d.first_booking_id}).")
        print(f"   Bookings made before that moment never have it; {d.docs_without_it_since_then} valid booking(s) made since then do not have it.")
    print("Handling: it is not required by the contract, so a booking without it stays valid; in `bookings` it is a column that is NULL where the document lacks the field (or has it as null).")

,field,docs_with_field,docs_total,first_booked_at,first_booking_id,docs_without_it_since_then
0,promo_code,173,259,2026-10-20 01:05:00+00:00,B0125,0


'promo_code' is in 173 of 259 valid bookings, first appearing on 2026-10-20 (01:05 UTC, B0125).
   Bookings made before that moment never have it; 0 valid booking(s) made since then do not have it.
Handling: it is not required by the contract, so a booking without it stays valid; in `bookings` it is a column that is NULL where the document lacks the field (or has it as null).


Part C: Model and store (DuckDB)

Star schema, fact grain = one ticket entry:
    -dim_event (one row per event, surrogate event_key)
    -dim_student (one row per student, surrogate student_key)
    -fact_ticket_sales: booking_id, event_key, student_key, ticket_type, qty, price, amount, status, date_key

The booking-level total is deliberately not in the fact (README Part D).
Surrogate keys are row_number() over the natural id. If an event or student id came with conflicting attributes in different bookings, the dimension keeps the earliest booking's values (the check below reports it).

In [11]:
con = duckdb.connect()                       # in-memory
con.execute("SET TimeZone = 'UTC'")          # date keys are computed in UTC, whatever the machine's timezone
con.register("bookings_df", bookings[CORE])
con.register("tickets_df", booking_tickets)

con.execute("""
CREATE TABLE dim_event AS
SELECT row_number() OVER (ORDER BY event_id) AS event_key, *
FROM (SELECT event_id, event_title AS title, event_club AS club,
             CAST(event_date AS DATE) AS event_date, event_venue AS venue
      FROM bookings_df
      QUALIFY row_number() OVER (PARTITION BY event_id ORDER BY booked_at, booking_id) = 1)
""")

con.execute("""
CREATE TABLE dim_student AS
SELECT row_number() OVER (ORDER BY student_id) AS student_key, *
FROM (SELECT student_id, student_name AS name, student_faculty AS faculty
      FROM bookings_df
      QUALIFY row_number() OVER (PARTITION BY student_id ORDER BY booked_at, booking_id) = 1)
""")

con.execute("""
CREATE TABLE fact_ticket_sales AS
SELECT b.booking_id,
       e.event_key,
       s.student_key,
       t.type AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS date_key
FROM tickets_df t
JOIN bookings_df b   USING (booking_id)
JOIN dim_event e     ON e.event_id = b.event_id
JOIN dim_student s   ON s.student_id = b.student_id
ORDER BY b.booking_id
""")

for tbl in ["dim_event", "dim_student", "fact_ticket_sales"]:
    n = con.execute(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {n} rows")
display(con.execute("SELECT * FROM dim_event ORDER BY event_key").df())
display(con.execute("SELECT * FROM dim_student ORDER BY student_key LIMIT 5").df())
display(con.execute("SELECT * FROM fact_ticket_sales LIMIT 5").df())

# Data-quality note: do ids map to a single set of attributes?
ev_conf = con.execute("SELECT count(*) FROM (SELECT event_id FROM bookings_df GROUP BY event_id HAVING count(DISTINCT (event_title, event_club, event_date, event_venue)) > 1)").fetchone()[0]
st_conf = con.execute("SELECT count(*) FROM (SELECT student_id FROM bookings_df GROUP BY student_id HAVING count(DISTINCT (student_name, student_faculty)) > 1)").fetchone()[0]
print(f"events with conflicting attributes: {ev_conf} | students with conflicting attributes: {st_conf}")

dim_event: 8 rows
dim_student: 103 rows
fact_ticket_sales: 314 rows


,event_key,event_id,title,club,event_date,venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


,student_key,student_id,name,faculty
0,1,58-1324,Nour Fouad,MET
1,2,58-1449,Farida Mansour,EMS
2,3,58-1801,Karim Kamel,IET
3,4,58-2165,Farida Hassan,Management
4,5,58-2307,Farida Hassan,Pharmacy


,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,date_key
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101


events with conflicting attributes: 0 | students with conflicting attributes: 0


Checks

In [12]:
fact_rows = con.execute("SELECT count(*) FROM fact_ticket_sales").fetchone()[0]
assert fact_rows == len(booking_tickets), (fact_rows, len(booking_tickets))

fact_amount = con.execute("SELECT sum(amount) FROM fact_ticket_sales").fetchone()[0]
total_sum = bookings["total"].sum()
assert math.isclose(fact_amount, total_sum, abs_tol=0.01), (fact_amount, total_sum)

# No ticket lost to a failed join, and keys are unique
assert con.execute("SELECT count(*) FROM fact_ticket_sales WHERE event_key IS NULL OR student_key IS NULL").fetchone()[0] == 0
assert con.execute("SELECT count(*) = count(DISTINCT event_key) FROM dim_event").fetchone()[0]
assert con.execute("SELECT count(*) = count(DISTINCT student_key) FROM dim_student").fetchone()[0]
assert con.execute("SELECT min(event_key) = 1 AND max(event_key) = count(*) FROM dim_event").fetchone()[0]
assert con.execute("SELECT min(student_key) = 1 AND max(student_key) = count(*) FROM dim_student").fetchone()[0]
print(f"OK: fact has {fact_rows} rows (= booking_tickets); sum(amount) = {fact_amount:,.2f} = sum(total) = {total_sum:,.2f}")

OK: fact has 314 rows (= booking_tickets); sum(amount) = 31,285.00 = sum(total) = 31,285.00


Answer from the star, confirmed bookings only, tickets sold and revenue per club and ticket type

In [13]:
answer = con.execute("""
SELECT e.club,
       f.ticket_type,
       CAST(sum(f.qty) AS BIGINT) AS tickets_sold,
       round(sum(f.amount), 2) AS revenue_egp
FROM fact_ticket_sales f
JOIN dim_event e USING (event_key)
WHERE f.status = 'confirmed'
GROUP BY e.club, f.ticket_type
ORDER BY e.club, f.ticket_type
""").df()
display(answer)
print(f"Confirmed: {answer.tickets_sold.sum()} tickets, {answer.revenue_egp.sum():,.2f} EGP")

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45,900.0
1,Chess,student,44,440.0
2,Debate,standard,65,1950.0
3,Debate,student,34,510.0
4,Debate,vip,40,3000.0
5,Football,standard,64,1600.0
6,Music,standard,67,2930.0
7,Music,student,20,1000.0
8,Music,vip,78,9990.0
9,Robotics,standard,60,2400.0


Confirmed: 577 tickets, 28,320.00 EGP


Save as Parquet
star/dim_event.parquet, star/dim_student.parquet, and the fact partitioned by the month the booking was made: star/fact_ticket_sales/booked_month=YYYY-MM/…,
(The month comes from date_key, which is the booking date in UTC.)

In [14]:
STAR = Path("star")
shutil.rmtree(STAR, ignore_errors=True)       # so re-running never leaves stale files
STAR.mkdir()

con.execute("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT parquet)")
con.execute("COPY dim_student TO 'star/dim_student.parquet' (FORMAT parquet)")
con.execute("""
COPY (SELECT *,
             substr(CAST(date_key AS VARCHAR), 1, 4) || '-' || substr(CAST(date_key AS VARCHAR), 5, 2) AS booked_month
      FROM fact_ticket_sales)
TO 'star/fact_ticket_sales' (FORMAT parquet, PARTITION_BY (booked_month))
""")

# Read it back to prove it round-trips
back = con.execute("SELECT count(*), sum(amount) FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)").fetchone()
assert back[0] == fact_rows and math.isclose(back[1], fact_amount, abs_tol=0.01)
for p in sorted(STAR.rglob("*.parquet")):
    print(p.as_posix())
print("Read back from Parquet:", back[0], "fact rows, sum(amount) =", round(back[1], 2))

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet
Read back from Parquet: 314 fact rows, sum(amount) = 31285.0


Numbers for the Part D paragraph (all computed)

In [15]:
print("lines read:", n_read)
print("quarantined by reason:", {k: int(v) for k, v in reason_counts.items()})
print("bookings left:", len(bookings), "| ticket rows:", len(booking_tickets))
print("same counts if dedupe ran first?", bool((cmp_.iloc[:, 0] == cmp_.iloc[:, 1]).all()))
if not drift.empty:
    print("drift:", drift_summary[["field", "first_booked_at"]].to_dict("records"))
over = con.execute("SELECT round(sum(total), 2) FROM bookings_df").fetchone()[0]
naive = con.execute("""
SELECT round(sum(b.total), 2)
FROM fact_ticket_sales f JOIN bookings_df b USING (booking_id)""").fetchone()[0]
print(f"sum(total) over bookings = {over}  vs  SUM(total) if it were repeated on each fact row = {naive}")

lines read: 283
quarantined by reason: {'tickets.type: not an allowed value': 3, 'tickets.qty: below the minimum (1)': 3, 'student.id: wrong format': 4, 'duplicate booking_id': 3, 'event: missing or null': 2, 'total does not equal sum of qty x price': 4, 'booked_at is not before the event date': 3, 'tickets: needs at least 1': 2}
bookings left: 259 | ticket rows: 314
same counts if dedupe ran first? True


drift: [{'field': 'promo_code', 'first_booked_at': Timestamp('2026-10-20 01:05:00+0000', tz='UTC')}]
sum(total) over bookings = 31285.0  vs  SUM(total) if it were repeated on each fact row = 42000.0
